# MiniGPT v3 â€” Colab Training Notebook

**Before running:** Runtime â†’ Change runtime type â†’ T4 GPU

**What this notebook does:**
1. Installs dependencies
2. Writes `model.py` inline (no upload needed)
3. Builds a QA-only corpus (no FineWeb download needed)
4. Trains a BPE tokenizer
5. Trains MiniGPTv2 for 5,000 steps on T4
6. Saves `my_minigpt_v3.pth` and `my_bpe_tokenizer_v3.json`
7. Downloads both files to your laptop

**Estimated time on T4:** ~2-3 minutes for 5,000 steps

In [ ]:
# Cell 1 â€” Install dependencies
!pip install -q tokenizers datasets

In [ ]:
# Cell 2 â€” Confirm GPU
import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Training device:', device)

In [ ]:
# Cell 3 â€” Write model.py
model_src = '''
import torch
import torch.nn as nn
from torch.nn import functional as F


class CausalHead(nn.Module):
    def __init__(self, n_embd, head_size, block_size, dropout):
        super().__init__()
        self.key   = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer("mask", torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        k, q = self.key(x), self.query(x)
        scores = q @ k.transpose(-2, -1) * (k.size(-1) ** -0.5)
        scores = scores.masked_fill(self.mask[:T, :T] == 0, float("-inf"))
        return self.dropout(F.softmax(scores, dim=-1)) @ self.value(x)


class MultiHead(nn.Module):
    def __init__(self, n_embd, n_head, block_size, dropout):
        super().__init__()
        head_size = n_embd // n_head
        self.heads = nn.ModuleList(
            [CausalHead(n_embd, head_size, block_size, dropout) for _ in range(n_head)]
        )
        self.proj    = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.dropout(self.proj(torch.cat([h(x) for h in self.heads], dim=-1)))


class FeedForward(nn.Module):
    def __init__(self, n_embd, dropout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.GELU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)


class TransformerBlock(nn.Module):
    def __init__(self, n_embd, n_head, block_size, dropout):
        super().__init__()
        self.attn = MultiHead(n_embd, n_head, block_size, dropout)
        self.ff   = FeedForward(n_embd, dropout)
        self.ln1  = nn.LayerNorm(n_embd)
        self.ln2  = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.ff(self.ln2(x))
        return x


class MiniGPTv2(nn.Module):
    def __init__(self, vocab_size, n_embd=128, n_head=4, n_layer=4,
                 block_size=256, dropout=0.1):
        super().__init__()
        self.block_size = block_size
        self.vocab_size = vocab_size
        self.token_embedding    = nn.Embedding(vocab_size, n_embd)
        self.position_embedding = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(
            *[TransformerBlock(n_embd, n_head, block_size, dropout) for _ in range(n_layer)]
        )
        self.ln_f   = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        x = self.token_embedding(idx) + self.position_embedding(
            torch.arange(T, device=idx.device)
        )
        x = self.blocks(x)
        logits = self.lm_head(self.ln_f(x))
        loss = None
        if targets is not None:
            loss = F.cross_entropy(
                logits.reshape(-1, self.vocab_size), targets.reshape(-1)
            )
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens=200, temperature=0.8):
        self.eval()
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / max(temperature, 0.01)
            probs  = F.softmax(logits, dim=-1)
            idx    = torch.cat([idx, torch.multinomial(probs, num_samples=1)], dim=1)
        return idx
'''

with open('model.py', 'w') as f:
    f.write(model_src)
print('model.py written.')

In [ ]:
# Cell 4 - Build corpus: QA-only (no FineWeb)
# Strategy: train exclusively on QA pairs so the model reliably learns
# question->answer mappings before any general-purpose text is added.
# The small corpus trains fast and the question-answer mapping is tight.

QA_PAIRS = [
    ('What is AI?', 'AI is a field of computing focused on systems that perform tasks associated with human intelligence.'),
    ('What is a computer?', 'A computer is a machine that processes data according to instructions.'),
    ('What is Python?', 'Python is a programming language known for its readable syntax.'),
    ('What is Python used for?', 'Python is used for automation, data analysis, machine learning, web development, and many other tasks.'),
    ('What is the internet?', 'The internet is a global network connecting computers and other devices.'),
    ('What is a language model?', 'A language model learns patterns in text and predicts what text is likely to come next.'),
    ('How does a language model learn?', 'A language model adjusts its weights during training to reduce errors in predicting the next token.'),
    ('What is a large language model?', 'A large language model is a neural network trained on large amounts of text to predict and generate language.'),
    ('What is machine learning?', 'Machine learning uses data to train systems to identify patterns and make predictions.'),
    ('What is deep learning?', 'Deep learning is machine learning that uses neural networks with multiple layers.'),
    ('What is a neural network?', 'A neural network is a computing model made of connected numerical units that learn patterns.'),
    ('What is a Transformer?', 'A Transformer is a neural network architecture that uses attention to process sequences.'),
    ('What is attention in a Transformer?', 'Attention lets a model weigh information from different positions in a sequence.'),
    ('What is training?', 'Training adjusts a model parameters to reduce prediction errors.'),
    ('What are model weights?', 'Model weights are learned numerical parameters that influence a neural network predictions.'),
    ('What is a token?', 'A token is a unit of text processed by a language model.'),
    ('What is a dataset?', 'A dataset is a collection of examples used for training or evaluating a model.'),
    ('What is overfitting?', 'Overfitting occurs when a model learns its training examples too specifically and performs poorly on new examples.'),
    ('What is a loss function?', 'A loss function measures how far a model predictions are from the target values.'),
    ('What is gradient descent?', 'Gradient descent updates model parameters in a direction intended to reduce the loss.'),
    ('What is backpropagation?', 'Backpropagation calculates gradients used to update neural network parameters.'),
    ('What is an embedding?', 'An embedding represents an item such as a token as a vector of numbers.'),
    ('What is RAG?', 'Retrieval-Augmented Generation retrieves relevant external information and supplies it as context for generating an answer.'),
    ('What is PyTorch?', 'PyTorch is a machine learning library that provides tensors, automatic differentiation, and neural network tools.'),
    ('What is a GPU?', 'A GPU is a processor designed to perform many operations in parallel and is widely used for neural network training.'),
    ('What is RAM?', 'RAM is temporary working memory used by a computer while programs are running.'),
    ('What is the CPU?', 'The CPU executes instructions and coordinates many general-purpose computing operations.'),
    ('What is gravity?', 'Gravity is the interaction that attracts objects with mass toward one another.'),
    ('What is water?', 'Water is a chemical compound made of hydrogen and oxygen.'),
    ('Why is the sky blue?', 'The sky appears blue because air scatters shorter blue wavelengths of sunlight more strongly than longer red wavelengths.'),
    ('What is the capital of France?', 'Paris is the capital of France.'),
    ('What is the capital of Japan?', 'Tokyo is the capital of Japan.'),
    ('What is Java?', 'Java is a general-purpose programming language commonly used for enterprise applications, backend systems, and Android development.'),
    ('What is the meaning of Java?', 'Java is the name of a programming language. The word can also refer to the Indonesian island of Java.'),
    ('Who is Trump?', 'Donald Trump is an American politician and businessman who has served as president of the United States.'),
    ('What is memory in a chatbot?', 'Memory can store information from earlier messages so a chatbot can use it later.'),
    ('What is your name?', 'I am MiniGPT, a small language model built as a local AI project.'),
    ('Who created you?', 'I was created by a developer as a local AI project using PyTorch.'),
    ('What can you do?', 'I can answer questions and generate text based on patterns learned during training.'),
    ('Hello', 'Hello! What would you like to ask?'),
    ('Hi', 'Hi! Ask me a question.'),
    ('How are you?', 'I am a small language model ready to respond to your message.'),
    ('Thank you', 'You are welcome.'),
    ('What is 2 plus 2?', '2 plus 2 equals 4.'),
    ('What is the solar system?', 'The solar system consists of the Sun and the objects gravitationally bound to it, including planets, moons, and comets.'),
    ('What is photosynthesis?', 'Photosynthesis is the process by which plants use light energy to make energy-rich compounds.'),
    ('What is electricity?', 'Electricity involves the presence or movement of electric charge.'),
    ('What is a variable in programming?', 'A variable is a named reference used to store or represent a value in a program.'),
    ('What is a function in programming?', 'A function is a reusable block of code that performs a defined task.'),
    ('What is a bug in software?', 'A bug is an error or defect that causes software to behave incorrectly.'),
    ('What is an algorithm?', 'An algorithm is a sequence of steps designed to solve a problem or perform a computation.'),
    ('What is hallucination in AI?', 'An AI hallucination is an output that presents incorrect or unsupported information as if it were true.'),
    ('Can a small language model answer every question?', 'No. Its capabilities are limited by its architecture, training, context, and available information.'),
]

# Each pair formatted as a conversation turn with [EOS] stop signal.
# Repeated 500x so the model sees each pair ~500 times during training.
single_pass = '\n\n'.join(
    f'User: {q}\nAssistant: {a} [EOS]'
    for q, a in QA_PAIRS
)
full_text = (single_pass + '\n\n') * 500

print(f'QA pairs         : {len(QA_PAIRS)}')
print(f'Repetitions      : 500')
print(f'Total characters : {len(full_text):,}')
print(f'Sample:')
print(full_text[:200])


In [ ]:
# Cell 5 â€” Train BPE tokenizer
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders

VOCAB_SIZE_TARGET = 2000

tok = Tokenizer(models.BPE(unk_token='[UNK]'))
tok.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
tok.decoder       = decoders.ByteLevel()

trainer = trainers.BpeTrainer(
    vocab_size=VOCAB_SIZE_TARGET,
    min_frequency=2,
    special_tokens=['[PAD]', '[UNK]', '[BOS]', '[EOS]'],
)
tok.train_from_iterator(
    (full_text[i : i + 100_000] for i in range(0, len(full_text), 100_000)),
    trainer=trainer,
    length=(len(full_text) + 99_999) // 100_000,
)

TOKENIZER_PATH = 'my_bpe_tokenizer_v3.json'
tok.save(TOKENIZER_PATH)
vocab_size = tok.get_vocab_size()
print(f'Tokenizer saved: {TOKENIZER_PATH}')
print(f'Actual vocab size: {vocab_size}')

# Quick sanity check
sample = tok.encode('User: hello\nAssistant:')
print(f'Prompt encode/decode: {tok.decode(sample.ids)!r}')

In [ ]:
# Cell 6 â€” Encode corpus and split 90/10
import torch

print('Encoding corpus ...')
all_ids = tok.encode(full_text).ids
data    = torch.tensor(all_ids, dtype=torch.long)

split_idx  = int(0.9 * len(data))
train_data = data[:split_idx]
val_data   = data[split_idx:]

print(f'  Total tokens     : {len(data):,}')
print(f'  Training tokens  : {len(train_data):,}  (90%)')
print(f'  Validation tokens: {len(val_data):,}  (10%)')

In [ ]:
# Cell 7 - Hyperparameters for QA-only training
# Small corpus -> smaller block_size, more steps to compensate,
# lower LR for stable convergence on repeated data.
import torch
from model import MiniGPTv2

BLOCK_SIZE = 128   # QA pairs are short; 128 covers most full turns
BATCH_SIZE = 32
N_EMBD     = 128
N_HEAD     = 4
N_LAYER    = 4
DROPOUT    = 0.1
MAX_ITERS  = 5000
EVAL_INTERVAL       = 500
CHECKPOINT_INTERVAL = 1000
LR = 1e-3   # higher LR is fine for small corpus memorisation

torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed(42)

model = MiniGPTv2(
    vocab_size=vocab_size,
    n_embd=N_EMBD,
    n_head=N_HEAD,
    n_layer=N_LAYER,
    block_size=BLOCK_SIZE,
    dropout=DROPOUT,
).to(device)

params = sum(p.numel() for p in model.parameters())
print(f'Device     : {device}')
print(f'Parameters : {params:,}')
print(f'vocab_size : {vocab_size}')
print(f'block_size : {BLOCK_SIZE}')
print(f'MAX_ITERS  : {MAX_ITERS}')
print(f'LR         : {LR}')


In [ ]:
# Cell 8 â€” Helper functions
import torch

def get_batch(split):
    source = train_data if split == 'train' else val_data
    ix = torch.randint(len(source) - BLOCK_SIZE - 1, (BATCH_SIZE,))
    x  = torch.stack([source[i     : i + BLOCK_SIZE    ] for i in ix])
    y  = torch.stack([source[i + 1 : i + BLOCK_SIZE + 1] for i in ix])
    return x.to(device), y.to(device)


@torch.no_grad()
def estimate_loss(n_batches=20):
    model.eval()
    results = {}
    for split in ('train', 'val'):
        losses = [model(*get_batch(split))[1].item() for _ in range(n_batches)]
        results[split] = sum(losses) / len(losses)
    model.train()
    return results


def save_checkpoint(step):
    path = 'my_minigpt_v3.pth'
    torch.save({
        'version':          'v3',
        'vocab_size':       vocab_size,
        'tokenizer_file':   'my_bpe_tokenizer_v3.json',
        'block_size':       BLOCK_SIZE,
        'n_embd':           N_EMBD,
        'n_head':           N_HEAD,
        'n_layer':          N_LAYER,
        'dropout':          DROPOUT,
        'step':             step,
        'model_state_dict': model.state_dict(),
    }, path)
    print(f'  Checkpoint saved: {path}  (step={step})')


def quick_generate(prompt, max_new=60, temperature=0.8):
    model.eval()
    ids     = tok.encode(prompt).ids
    context = torch.tensor([ids], dtype=torch.long, device=device)
    with torch.no_grad():
        out = model.generate(context, max_new_tokens=max_new, temperature=temperature)
    new_ids = out[0][len(ids):].tolist()
    model.train()
    return tok.decode(new_ids)


print('Helper functions ready.')

In [ ]:
# Cell 9 - Training loop (5,000 steps, ~2-3 min on T4)
import time

optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
model.train()

t0 = time.time()
print(f'Training for {MAX_ITERS} steps ...')
print(f'{"Step":>6}  {"train":>7}  {"val":>7}  {"elapsed":>8}')
print('-' * 38)

for step in range(MAX_ITERS):
    xb, yb = get_batch('train')
    _, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()

    if step % EVAL_INTERVAL == 0:
        scores  = estimate_loss()
        elapsed = time.time() - t0
        print(f'{step:>6}  {scores["train"]:>7.3f}  {scores["val"]:>7.3f}  {elapsed:>7.1f}s')

    if step > 0 and step % CHECKPOINT_INTERVAL == 0:
        save_checkpoint(step)

# Final save
save_checkpoint(MAX_ITERS - 1)
print(f'\nTotal training time: {time.time() - t0:.1f}s')

In [ ]:
# Cell 10 â€” Post-training generation samples
test_prompts = [
    'User: hello\nAssistant:',
    'User: What is AI?\nAssistant:',
    'User: What is Python?\nAssistant:',
    'User: What is your name?\nAssistant:',
    'User: the hospital needs medical supplies because\nAssistant:',
]

print('=== Generation samples ===')
for p in test_prompts:
    out = quick_generate(p, max_new=80, temperature=0.7)
    # Trim at next User: turn
    for marker in ['\n\nUser:', '\nUser:']:
        if marker in out:
            out = out.split(marker)[0]
    print(f'\nPrompt : {p!r}')
    print(f'Output : {out.strip()!r}')

In [ ]:
# Cell 11 â€” Verify matched pair before download
import json
from tokenizers import Tokenizer as HFTokenizer

saved = torch.load('my_minigpt_v3.pth', map_location='cpu', weights_only=False)
tok2  = HFTokenizer.from_file('my_bpe_tokenizer_v3.json')

assert saved['vocab_size'] == tok2.get_vocab_size(), (
    f"MISMATCH: checkpoint={saved['vocab_size']} tokenizer={tok2.get_vocab_size()}"
)
assert saved['tokenizer_file'] == 'my_bpe_tokenizer_v3.json'

print('Verification passed:')
print(f'  vocab_size     : {saved["vocab_size"]}  (checkpoint == tokenizer)')
print(f'  tokenizer_file : {saved["tokenizer_file"]}')
print(f'  step           : {saved["step"]}')
print(f'  block_size     : {saved["block_size"]}')
print(f'  n_embd         : {saved["n_embd"]}')
print(f'  n_layer        : {saved["n_layer"]}')

In [ ]:
# Cell 12 â€” Download both files to your laptop
from google.colab import files
files.download('my_minigpt_v3.pth')
files.download('my_bpe_tokenizer_v3.json')
print('Download started for both files.')
print('Save them to:')
print('  checkpoints/my_minigpt_v3.pth')
print('  tokenizers/my_bpe_tokenizer_v3.json')